# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeremOzcn/flyrank-ml-internship-submission/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Completed task framing for Lane 2 using the bundled anonymized starter dataset. Run all cells from inside the repository; Python, pandas, and a standard Jupyter kernel are sufficient.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2 — Refresh / Content Opportunity Scoring; task type: ranking.** This continues the explicit choice in `w01_research_question.ipynb`. An SEO strategist needs an ordered queue of pages for editorial review, rather than just a yes/no flag or a cluster. A learned score could order that queue, but ranking is the business task.

For this exercise, I assume a team can review **50 pages per cycle across the portfolio**. It would inspect the highest-priority pages and decide whether to update outdated facts, expand missing coverage, revise metadata, or leave the page alone. False positives waste editorial time; false negatives leave potentially important declines unreviewed. The starter snapshot supports framing and retrospective checks, not a claim of future recovery.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works from the repository root or work/notebooks (and other repo subfolders).
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data/raw/content_refresh_anonymized.csv").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run from inside the repo with the starter CSV present.")
df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")
print("Source: data/raw/content_refresh_anonymized.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Pandas version:", pd.__version__)


Source: data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Pandas version: 2.3.3


## 2. Target or proxy

There is **no observed refresh-benefit target** in the CSV. I create `is_declining_proxy = (trend_direction == "down")`, matching the reference pipeline's label logic. The dictionary defines `down` as a greater-than-20% fall in impressions from the previous 30 days to the latest 30 days. This is a thresholded summary of observed traffic, used here as **binary ranking relevance**, not an editor-verified need to refresh.

My lane slice keeps `impressions_90d >= 100` (an explicit, provisional visibility floor) and `impressions_prev_30d > 0` (a defined comparison denominator). It retains both declining and non-declining pages and does not filter on staleness: recently updated pages can also need review. New/zero-history pages need a separate monitoring workflow. The 100-impression floor does not eliminate low-count volatility.

Decline is a reasonable triage proxy because losing existing visibility merits investigation. It can also reflect seasonality, demand changes, or measurement issues. A model fitted to this label might only reproduce its definition. If the goal is just to list already-declining pages, directly using the observed trend is sufficient. A later useful ML target would be independently reviewed priority or a decline measured after a feature cutoff; neither is available here.

In [2]:
required = ["content_id", "client_id", "impressions_90d",
            "impressions_prev_30d", "trend_direction"]
assert df[required].notna().all().all(), "Missing IDs or eligibility/label inputs"
assert df["trend_direction"].isin(["down", "stable", "up", "new", "flat"]).all()
assert df["content_id"].is_unique, "Expected one row per content item"

df["is_declining_proxy"] = df["trend_direction"].eq("down").astype("int8")
eligible = df.loc[
    df["impressions_90d"].ge(100) & df["impressions_prev_30d"].gt(0)
].copy()
print(f"All pages: {len(df):,}; clients: {df['client_id'].nunique()}")
print(f"Eligible pages: {len(eligible):,}; excluded: {len(df) - len(eligible):,}")
print(f"Proxy positives in slice: {eligible['is_declining_proxy'].sum():,} "
      f"({eligible['is_declining_proxy'].mean():.1%})")
display(eligible[["content_id", "impressions_90d", "impressions_prev_30d",
                  "impressions_last_30d", "trend_direction", "is_declining_proxy"]].head())


All pages: 30,000; clients: 32
Eligible pages: 21,758; excluded: 8,242
Proxy positives in slice: 13,152 (60.4%)


,content_id,impressions_90d,impressions_prev_30d,impressions_last_30d,trend_direction,is_declining_proxy
0,content_304f48230142,3803,987,578,down,1
1,content_a1fb4e703a9e,15320,5915,2501,down,1
2,content_9aa793d4d895,12581,6089,2382,down,1
3,content_331d6c4de07b,11751,4206,3626,stable,0
4,content_d99b7a2d90ca,19140,6452,4211,down,1


## 3. Success metric

**Precision@50 = proxy-positive pages among the first 50 recommendations / 50.** It measures how many of the team's limited review slots reach pages with observed decline. Overall accuracy would ignore the order of the queue. This metric weights each review slot equally; it does not measure recovered clicks or refresh ROI.

I compute a descriptive baseline now: put pages with `days_since_last_update >= 90` first, then sort by `impressions_90d` descending, breaking ties by `content_id`. For uniform random selection, expected Precision@50 equals the eligible slice's positive rate.

For a future model, I propose success as **at least 0.10 absolute improvement over this rule's Precision@50** (five additional relevant pages per 50), also exceeding the random expectation on the same held-out pool. This is a proposed acceptance threshold, not a measured model result. Use disjoint `client_id` groups for training and evaluation, choose settings only on training/validation groups, and report variation across held-out groups. These full-slice baseline numbers are not held-out validation; a forward-prediction claim additionally requires non-overlapping feature/outcome windows.

In [3]:
K = 50
assert len(eligible) >= K
rule_queue = (
    eligible.assign(stale_rule=eligible["days_since_last_update"].ge(90))
    .sort_values(["stale_rule", "impressions_90d", "content_id"],
                 ascending=[False, False, True])
)
rule_hits = int(rule_queue.head(K)["is_declining_proxy"].sum())
baseline_metrics = pd.DataFrame([
    {"method": "Stale-first, then impressions (observed)",
     "precision_at_50": rule_hits / K},
    {"method": "Uniform random selection (expected)",
     "precision_at_50": eligible["is_declining_proxy"].mean()},
])
display(baseline_metrics)
print(f"Rule queue: {rule_hits}/{K} pages have the decline proxy.")
print("Descriptive baseline only; no model has been trained or evaluated.")


,method,precision_at_50
0,"Stale-first, then impressions (observed)",0.440000
1,Uniform random selection (expected),0.604467


Rule queue: 22/50 pages have the decline proxy.
Descriptive baseline only; no model has been trained or evaluated.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page (`content_id`) belonging to one client (`client_id`), at the export snapshot.** It is not a daily observation or a search query. Activity columns summarize trailing windows. Uniqueness and client counts are checked in code.

`working_df` below narrows the eligible rows to identifiers, content context, review signals, and the proxy. Nulls are reported before cleaning. Following `docs/data-dictionary.md`, `avg_position == 0` becomes missing, and missing-value flags preserve that distinction. Unknown word counts and keyword demand remain missing rather than becoming zero. Missingness is also shown by content type. Rate columns remain percentage units: `ctr = 0.76` means **0.76%**.

**Leakage boundary:** IDs are only for joining/splitting. `trend_direction`, `trend_pct`, and the comparison-window impression inputs are excluded from the working signals because they reveal/reconstruct the label. Even the retained 90-day metrics overlap the observed decline window: they are current review context, **not certified predictors of future decline**. Before training a forward model, rebuild features from a strictly earlier window. Removing direct label columns alone does not fix temporal leakage.

In [4]:
context_columns = [
    "content_type", "main_intent", "search_volume", "word_count",
    "content_age_days", "days_since_last_update", "impressions_90d",
    "clicks_90d", "ctr", "avg_position", "engagement_rate",
]
working_columns = ["content_id", "client_id", *context_columns, "is_declining_proxy"]
quality_columns = [*working_columns, "trend_direction", "trend_pct",
                   "impressions_prev_30d", "impressions_last_30d"]
display(pd.DataFrame({
    "dtype": df[quality_columns].dtypes.astype(str),
    "nulls_all_pages": df[quality_columns].isna().sum(),
    "nulls_lane_slice": eligible[quality_columns].isna().sum(),
}))
print("Duplicate full rows:", int(df.duplicated().sum()))
print("Duplicate content IDs:", int(df["content_id"].duplicated().sum()))
print("Position=0 (unavailable), all / slice:",
      int(df["avg_position"].eq(0).sum()), int(eligible["avg_position"].eq(0).sum()))
print("Missing percentages by content type (eligible pages):")
display(eligible.groupby("content_type")[["search_volume", "word_count"]]
        .agg(lambda s: s.isna().mean() * 100).round(1))

working_df = eligible[working_columns].copy()
working_df["avg_position"] = working_df["avg_position"].mask(working_df["avg_position"].eq(0))
for column in ["search_volume", "word_count", "avg_position"]:
    working_df[f"has_{column}"] = working_df[column].notna()
assert working_df["content_id"].is_unique
assert not {"trend_direction", "trend_pct", "impressions_last_30d",
            "impressions_prev_30d"}.intersection(working_df.columns)
print("Working dataframe shape:", working_df.shape)
with pd.option_context("display.max_columns", None):
    display(working_df.head())


,dtype,nulls_all_pages,nulls_lane_slice
content_id,object,0,0
client_id,object,0,0
content_type,object,0,0
main_intent,object,2374,486
search_volume,float64,2468,557
word_count,float64,7699,6531
content_age_days,int64,0,0
days_since_last_update,int64,0,0
impressions_90d,int64,0,0
clicks_90d,int64,0,0


Duplicate full rows: 0
Duplicate content IDs: 0
Position=0 (unavailable), all / slice: 1205 0
Missing percentages by content type (eligible pages):


,search_volume,word_count
content_type,,
comparison article,0.0,0.0
feedly article,100.0,0.0
keyword article,1.0,31.0


Working dataframe shape: (21758, 17)


,content_id,client_id,content_type,main_intent,search_volume,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,is_declining_proxy,has_search_volume,has_word_count,has_avg_position
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,10.0,3221.0,187,20,3803,29,0.76,10.6,5.88,1,True,True,True
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,90.0,2481.0,445,25,15320,7,0.05,20.3,0.00,1,True,True,True
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,0.0,3515.0,141,20,12581,11,0.09,36.5,0.00,1,True,True,True
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,10.0,NaN,463,22,11751,58,0.49,6.2,1.28,0,True,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,0.0,2803.0,263,14,19140,24,0.13,44.0,0.00,1,True,True,True


## 5. Why ML beats a fixed rule here

**ML could improve prioritization, but this notebook does not establish that it beats a rule.** The stale-first rule treats crossing 90 days as decisive. It can send stable old pages ahead of recently updated pages losing visibility, and ignores whether a low CTR accompanies poor position, weak demand, or a particular content type. The table below measures where that simple age flag agrees or disagrees with observed decline in this slice.

A learned ranking could combine demand, freshness, content type, and engagement patterns rather than hand-tuning every interaction. It earns a place only if it improves held-out Precision@50 against the fixed rule using an appropriate target and feature window. Training on trend or its component counts would just rediscover a known rule and add no value.

The intended output is a queue with page IDs and review reasons, used by an editor to inspect facts, coverage, and metadata before choosing an action. A decline flag alone must not trigger an automatic rewrite. This is decision support; neither the proxy nor these cross-sectional comparisons show that refreshing causes traffic recovery.

In [5]:
rule_comparison = (
    eligible.assign(stale_rule=eligible["days_since_last_update"].ge(90))
    .groupby("stale_rule")["is_declining_proxy"]
    .agg(pages="size", declining_pages="sum", decline_rate="mean")
    .rename(index={False: "Updated <90 days ago", True: "Updated >=90 days ago"})
)
rule_comparison["non_declining_pages"] = (
    rule_comparison["pages"] - rule_comparison["declining_pages"]
)
display(rule_comparison)
print("Observed proxy agreement, not verified refresh need or model performance.")


,pages,declining_pages,decline_rate,non_declining_pages
stale_rule,,,,
Updated <90 days ago,13654,8094,0.592793,5560
Updated >=90 days ago,8104,5058,0.624136,3046


Observed proxy agreement, not verified refresh need or model performance.


## Self-check

- [x] Preserved all template sections in order, with markdown reasoning and executed code.
- [x] Continued Week 1's lane and specified ranking, a concrete proxy, and Precision@50.
- [x] Loaded the actual starter CSV; showed shape, columns, quality checks, and a working dataframe sample.
- [x] Verified one row per page and displayed engineered proxy values.
- [x] Measured a fixed-rule baseline and explained the editorial action and ML limitations.
- [x] Distinguished descriptive results from held-out performance, future prediction, and causal refresh benefit.
- [x] Ran top to bottom without errors and saved visible outputs; samples contain only anonymized starter data.
- [ ] Commit `work/notebooks/w02_ml_task_framing.ipynb`, then submit the repository URL on ML-03.

Local references: `skills/framing-ml-problems/SKILL.md`, `skills/flyrank/flyrank-data/SKILL.md`, `docs/data-dictionary.md`, `scripts/01_prepare_features.py`, and `work/notebooks/w01_research_question.ipynb`.
